In [1]:
from qat.lang.AQASM import Program, H, X, CNOT, RX, I, RY, RZ, CSIGN #Gates
from qat.core import Observable, Term, Batch #Hamiltonian
import numpy as np
from qat.plugins import ScipyMinimizePlugin
import pickle

/home/harshit-verma/Desktop/eviden/eviden/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nqbt = 5 # number of qubits
nruns = 0 #nbshots for observable sampling
nseeds = 50

In [3]:
dep = np.arange(1, 11, 1, dtype = int)

In [4]:
#Instantiation of Hamiltoniian
heisen = Observable(nqbt)
#Generation of Heisenberg Hamiltonian
for q_reg in range(nqbt-1):
    heisen += Observable(nqbt, pauli_terms = [Term(1., typ, [q_reg,q_reg + 1]) for typ in ['XX','YY','ZZ']])

In [5]:
#exact calculation for ground state
from qat.fermion import SpinHamiltonian

heisen_class = SpinHamiltonian(nqbits=heisen.nbqbits, terms=heisen.terms)
heisen_mat = heisen_class.get_matrix()

eigvals, eigvecs = np.linalg.eigh(heisen_mat)
g_energy = eigvals[0]
g_state = eigvecs[:,0]

In [6]:
from qat.qpus import get_default_qpu
qpu_ideal = get_default_qpu()

In [7]:
ress = {}

In [8]:
#variational gates
for ht in range(nseeds):
    np.random.seed(10*ht)
    res = []
    for ct in dep:
        qprog = Program()
        qbits = qprog.qalloc(nqbt)
        #variational parameters used for generating gates
        angs = [qprog.new_var(float, 'a_%s'%i) for i in range(nqbt*(2*ct+1))]
        ang = iter(angs)
        #circuit
        for it in range(ct):
            for q_index in range(nqbt):
                RY(next(ang))(qbits[q_index])
            for q_index in range(nqbt):
                if not q_index%2 and q_index <= nqbt-2:
                    CSIGN(qbits[q_index],qbits[q_index+1])
            for q_index in range(nqbt):
                RY(next(ang))(qbits[q_index])
            for q_index in range(nqbt):
                if q_index%2 and q_index <= nqbt-2:
                    CSIGN(qbits[q_index],qbits[q_index+1])
            CSIGN(qbits[0],qbits[nqbt-1])
            if it==(ct-1):
                for q_index in range(nqbt):
                    RY(next(ang))(qbits[q_index])
        #circuit
        circuit = qprog.to_circ()
        optimizer_scipy = ScipyMinimizePlugin(method="COBYLA",
                                  tol=1e-6,
                                  options={"maxiter": 25000},
#                                  x0=np.zeros(nqbt)
                                     )
        stack_noiseless = optimizer_scipy | qpu_ideal
        #job at hand, not the observable class but instantiated job
        job = circuit.to_job(observable = heisen, nbshots = 0)
        res_noiseless = stack_noiseless.submit(job)
        res.append(res_noiseless)
    ress['%s'%ht] = res

In [9]:
#with open('global_noisy_seeds/noisy-%s_%s.pkl'%(base, expo), 'wb') as file:
with open('RYA/Results/noiseless_COBY_50_seeds.pkl', 'wb') as file:
    pickle.dump(ress, file)